In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_DIR = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'players.csv').exists()), Path.cwd())
DATA_DIR = PROJECT_DIR / 'data'
MODEL_DIR = PROJECT_DIR / 'models'
REPORT_DIR = PROJECT_DIR / 'reports'
MODEL_DIR.mkdir(exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)
LEAGUE = "GB1"
START_DATE = pd.Timestamp("2013-01-01")
TEST_CUTOFF = pd.Timestamp("2023-07-01")
print("Data directory:", DATA_DIR)
print("pandas:", pd.__version__, "scikit-learn:", sklearn.__version__)


Data directory: C:\Users\Asus\Desktop\PL project
pandas: 2.3.3 scikit-learn: 1.9.1


In [2]:
valuation_columns = [
    "player_id", "date", "market_value_in_eur", "current_club_name",
    "current_club_id", "player_club_domestic_competition_id"
]
vals = pd.read_csv(DATA_DIR / "player_valuations.csv", usecols=valuation_columns, parse_dates=["date"])
vals = vals.loc[
    vals["player_club_domestic_competition_id"].eq(LEAGUE)
    & vals["date"].ge(START_DATE)
    & vals["market_value_in_eur"].gt(0)
].sort_values(["date", "player_id"], kind="mergesort").reset_index(drop=True)

player_columns = [
    "player_id", "date_of_birth", "position", "sub_position", "foot",
    "height_in_cm", "country_of_citizenship"
]
player_info = pd.read_csv(DATA_DIR / "players.csv", usecols=player_columns, parse_dates=["date_of_birth"])
appearance_columns = [
    "appearance_id", "player_id", "date", "goals", "assists",
    "minutes_played", "yellow_cards", "red_cards"
]
apps = pd.read_csv(DATA_DIR / "appearances.csv", usecols=appearance_columns, parse_dates=["date"])
print(f"Eligible Premier League valuations: {len(vals):,} ({vals.date.min().date()} to {vals.date.max().date()})")
print(f"Appearance records loaded: {len(apps):,}")


Eligible Premier League valuations: 28,313 (2013-01-04 to 2026-06-03)
Appearance records loaded: 1,894,350


In [3]:
# Build cumulative appearance totals, then query only strictly before each valuation date.
# The lower-bound query is also strict, producing a clean trailing 365-day window.
performance_columns = ["goals", "assists", "minutes_played", "yellow_cards", "red_cards", "games"]
apps = apps.dropna(subset=["player_id", "date"]).copy()
apps["games"] = 1
apps = apps.sort_values(["date", "player_id", "appearance_id"], kind="mergesort").reset_index(drop=True)
cumulative = apps.groupby("player_id", sort=False)[performance_columns].cumsum()
history = pd.concat([apps[["player_id", "date"]], cumulative], axis=1).sort_values("date", kind="mergesort")
queries = vals[["player_id", "date"]].copy()
queries["_row"] = np.arange(len(queries))

def cumulative_asof(query_dates):
    left = queries[["_row", "player_id"]].copy()
    left["query_date"] = query_dates.to_numpy()
    left = left.sort_values("query_date", kind="mergesort")
    matched = pd.merge_asof(
        left, history, left_on="query_date", right_on="date", by="player_id", direction="backward"
    )
    return matched.set_index("_row")[performance_columns].reindex(queries.index).fillna(0)

one_day = pd.Timedelta(nanoseconds=1)
recent = cumulative_asof(vals["date"] - one_day)
prior = cumulative_asof(vals["date"] - pd.Timedelta(days=365) - one_day)
performance = (recent.to_numpy() - prior.to_numpy())
perf = pd.DataFrame(performance, columns=[f"{name}_12m" for name in performance_columns], index=vals.index)
print("Computed strictly prior 365-day performance features.")


Computed strictly prior 365-day performance features.


In [4]:
data = vals.join(perf).merge(player_info, on="player_id", how="left", validate="many_to_one")
data["age"] = (data["date"] - data["date_of_birth"]).dt.days / 365.25
data["minutes_per_game"] = data["minutes_played_12m"] / data["games_12m"].replace(0, np.nan)
data = data.dropna(subset=["age"]).copy()
data["target_log"] = np.log1p(data["market_value_in_eur"])
print(f"Model rows after requiring known age: {len(data):,}")
print(f"Temporal split: train before {TEST_CUTOFF.date()}, test on/after cutoff")


Model rows after requiring known age: 28,304
Temporal split: train before 2023-07-01, test on/after cutoff


In [5]:
numeric_full = [
    "age", "height_in_cm", "goals_12m", "assists_12m", "minutes_played_12m",
    "yellow_cards_12m", "red_cards_12m", "games_12m", "minutes_per_game"
]
categorical_full = ["position", "sub_position", "foot", "country_of_citizenship", "current_club_name"]
numeric_baseline = ["age", "height_in_cm"]
categorical_baseline = ["position", "current_club_name"]

train = data.loc[data["date"] < TEST_CUTOFF].copy()
test = data.loc[data["date"] >= TEST_CUTOFF].copy()
if train.empty or test.empty:
    raise ValueError("The chronological train/test split is empty. Check dates and cutoff.")

def make_model(numeric_features, categorical_features):
    preprocess = ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scale", StandardScaler())
        ]), numeric_features),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_features)
    ])
    return Pipeline([("preprocess", preprocess), ("regression", LinearRegression())])

print(f"Training rows: {len(train):,} | Holdout rows: {len(test):,}")


Training rows: 22,457 | Holdout rows: 5,847


In [6]:
def evaluate_model(label, estimator, features):
    estimator.fit(train[features], train["target_log"])
    predicted_log = estimator.predict(test[features])
    predicted_eur = np.maximum(np.expm1(predicted_log), 0)
    actual_eur = test["market_value_in_eur"].to_numpy()
    absolute_error = np.abs(predicted_eur - actual_eur)
    return {
        "model": label,
        "rows": int(len(test)),
        "rmse_eur": float(np.sqrt(mean_squared_error(actual_eur, predicted_eur))),
        "mae_eur": float(mean_absolute_error(actual_eur, predicted_eur)),
        "median_absolute_error_eur": float(np.median(absolute_error)),
        "median_absolute_percentage_error": float(np.median(absolute_error / actual_eur) * 100),
        "rmse_log": float(np.sqrt(mean_squared_error(test["target_log"], predicted_log))),
        "r2_log": float(r2_score(test["target_log"], predicted_log)),
    }

baseline_features = numeric_baseline + categorical_baseline
full_features = numeric_full + categorical_full
baseline_model = make_model(numeric_baseline, categorical_baseline)
full_model = make_model(numeric_full, categorical_full)
baseline_metrics = evaluate_model("Baseline: age, height, position, club", baseline_model, baseline_features)
full_metrics = evaluate_model("Retrained: profile plus prior-year performance", full_model, full_features)
comparison = [baseline_metrics, full_metrics]
print(json.dumps(comparison, indent=2))


[
  {
    "model": "Baseline: age, height, position, club",
    "rows": 5847,
    "rmse_eur": 21712007.900374677,
    "mae_eur": 13344311.344181474,
    "median_absolute_error_eur": 7248826.07684849,
    "median_absolute_percentage_error": 72.04368163413803,
    "rmse_log": 1.402459693122977,
    "r2_log": 0.13321083769341135
  },
  {
    "model": "Retrained: profile plus prior-year performance",
    "rows": 5847,
    "rmse_eur": 15655407.721114358,
    "mae_eur": 9723149.946012426,
    "median_absolute_error_eur": 5373983.084157213,
    "median_absolute_percentage_error": 55.63909149225441,
    "rmse_log": 1.032792648392449,
    "r2_log": 0.5299338573383148
  }
]


In [1]:
# Refit the candidate on all eligible historical observations only after the holdout report is fixed.
# The saved candidate is separate from the existing app model and existing joblib file.
import joblib

final_model = make_model(numeric_full, categorical_full)
final_model.fit(data[full_features], data["target_log"])
artifact = {
    "model": final_model,
    "numeric_features": numeric_full,
    "categorical_features": categorical_full,
    "league_competition_id": LEAGUE,
    "target": "log1p_market_value_in_eur",
    "training_rows": int(len(data)),
    "training_date_start": str(data["date"].min().date()),
    "training_date_end": str(data["date"].max().date()),
    "performance_window": "strictly prior 365 days",
    "club_features": "club name from each historical valuation; no current club roster snapshot",
}
joblib.dump(artifact, MODEL_DIR / "pl_value_model_retrained.joblib")
report = {
    "model": "linear_regression_log1p_with_prior_365d_performance",
    "league_competition_id": LEAGUE,
    "test_cutoff": str(TEST_CUTOFF.date()),
    "train_rows": int(len(train)),
    "test_rows": int(len(test)),
    "test_date_start": str(test["date"].min().date()),
    "test_date_end": str(test["date"].max().date()),
    "features": full_features,
    "metrics": comparison,
    "notes": [
        "Metrics are from the chronological holdout before the final all-data refit.",
        "RMSE and median percentage error are in euros and percent respectively.",
        "No current clubs.csv snapshot features are used for historical observations.",
        "Appearance features include only matches strictly before each valuation date.",
        "Player profile fields other than age are the available current player records.",
    ],
}
(REPORT_DIR / "model_evaluation_retrained.json").write_text(json.dumps(report, indent=2), encoding="utf-8")
print("Saved model: models/pl_value_model_retrained.joblib")
print("Saved report: reports/model_evaluation_retrained.json")


NameError: name 'make_model' is not defined